# Hidden Markov Models: implementation and verification

ALU intranet project 2331. This notebook exercises the seven numbered modules. It resumes the earlier hot/cold ice-cream calculation. It is a discrete HMM exercise, not the human-activity formative: no sensor recordings or activity-classification results are claimed here.

In [1]:
import importlib
import itertools
import numpy as np

markov_chain = importlib.import_module("0-markov_chain").markov_chain
regular = importlib.import_module("1-regular").regular
absorbing = importlib.import_module("2-absorbing").absorbing
forward = importlib.import_module("3-forward").forward
viterbi = importlib.import_module("4-viterbi").viterbi
backward = importlib.import_module("5-backward").backward
baum_welch = importlib.import_module("6-baum_welch").baum_welch

## 1. Evolving a Markov chain

A row distribution advances as s(t+1) = s(t)A. A stationary distribution satisfies pi A = pi. Regularity additionally requires that some power of A has all positive entries; a periodic chain can have a stationary distribution without being regular. Absorption instead asks whether every state can eventually reach an absorbing state.

In [2]:
A = np.array([[0.6, 0.4], [0.3, 0.7]])
s = np.array([[1.0, 0.0]])
print("After two transitions:", markov_chain(A, s, 2))
print("Stationary distribution:", regular(A))
print("Identity regular?", regular(np.eye(2)))
print("Identity absorbing?", absorbing(np.eye(2)))

After two transitions: [[0.48 0.52]]
Stationary distribution: [[0.42857143 0.57142857]]
Identity regular? None
Identity absorbing? True


## 2. Resume the ice-cream exercise

States 0 and 1 mean hot and cold. Observation indices 0, 1, 2 mean one, two, three ice creams. The sequence (3,1,3) is therefore encoded [2,0,2]. Forward sums over all eight paths; Viterbi takes their maximum. These quantities answer different questions.

In [3]:
A = np.array([[0.6, 0.4], [0.5, 0.5]])
E = np.array([[0.2, 0.4, 0.4], [0.5, 0.4, 0.1]])
pi = np.array([[0.8], [0.2]])
observations = np.array([2, 0, 2])
probabilities = {}
for path in itertools.product(range(2), repeat=3):
    p = pi[path[0], 0] * E[path[0], observations[0]]
    for t in range(1, 3):
        p *= A[path[t-1], path[t]] * E[path[t], observations[t]]
    label = "".join("H" if i == 0 else "C" for i in path)
    probabilities[label] = p
    print(label, format(p, ".6f"))
print("Sum:", format(sum(probabilities.values()), ".6f"))

HHH 0.009216
HHC 0.001536
HCH 0.012800
HCC 0.003200
CHH 0.000480
CHC 0.000080
CCH 0.001000
CCC 0.000250
Sum: 0.028562


## 3. Forward, backward, and decoding

Forward alpha(i,t) includes the prefix observations and current hidden state. Backward beta(i,t) gives the likelihood of the remaining observations conditional on that state. Their product, summed over states, must equal the full likelihood at every t. Viterbi keeps predecessor pointers and reconstructs a single complete path; choosing the locally largest forward value independently at each time would be incorrect.

In [4]:
likelihood, F = forward(observations, E, A, pi)
reverse_likelihood, B = backward(observations, E, A, pi)
path, best_probability = viterbi(observations, E, A, pi)
np.testing.assert_allclose(likelihood, 0.028562)
np.testing.assert_allclose(reverse_likelihood, likelihood)
np.testing.assert_allclose((F * B).sum(axis=0), likelihood)
assert path == [0, 1, 0]
np.testing.assert_allclose(best_probability, 0.0128)
print("Observation likelihood:", likelihood)
print("Best path:", " -> ".join("Hot" if i == 0 else "Cold" for i in path))
print("Best joint path probability:", best_probability)
print("Posterior probability of this path:", best_probability / likelihood)

Observation likelihood: 0.028562000000000008
Best path: Hot -> Cold -> Hot
Best joint path probability: 0.012799999999999997
Posterior probability of this path: 0.44814788880330486


## 4. Learning without observed states

Baum-Welch alternates posterior expected counts with probability re-estimation. Transition counts exclude the final time because no departure follows it; emission counts include every time. Scaling prevents the sequence probability from vanishing numerically during training. Initial probabilities remain fixed to match the interface, which returns A and E only. This small example illustrates likelihood improvement, not generalization performance.

In [5]:
learned_A, learned_E = A.copy(), E.copy()
last = likelihood
for iteration in range(1, 6):
    learned_A, learned_E = baum_welch(
        observations, learned_A, learned_E, pi, iterations=1)
    current, _ = forward(observations, learned_E, learned_A, pi)
    assert current + 1e-12 >= last
    print("EM step", iteration, "likelihood", current)
    last = current
np.testing.assert_allclose(learned_A.sum(axis=1), 1)
np.testing.assert_allclose(learned_E.sum(axis=1), 1)
print("Transition matrix:\n", learned_A)
print("Emission matrix:\n", learned_E)

EM step 1 likelihood 0.19621988557132242
EM step 2 likelihood 0.3367732978721262
EM step 3 likelihood 0.6699103528058065
EM step 4 likelihood 0.794778794927674
EM step 5 likelihood 0.79998011978908
Transition matrix:
 [[3.37406789e-11 1.00000000e+00]
 [9.99987575e-01 1.24251039e-05]]
Emission matrix:
 [[8.78060891e-11 0.00000000e+00 1.00000000e+00]
 [9.99987575e-01 0.00000000e+00 1.24252978e-05]]


## Verification and submission limits

The separate regression suite checks exhaustive paths in 30 random small models, the published Baum-Welch example, invalid inputs, impossible observations, periodic and disconnected chains, long-sequence training, and preservation of caller inputs. Run `python3 -m unittest discover -s tests -v` from this folder. The course specifies Python 3.5 / NumPy 1.15; local numerical verification used Python 3.9 / NumPy 2.0.2. The implementation uses syntax compatible with Python 3.5, but the original grader environment was not available locally. The public forward/backward trellises intentionally retain raw probabilities and can underflow on very long sequences.